In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\DR.KSS_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,259.0,164.0,181.0,106.0,91.0,96.0,69.0,92.0,140.0,146.0,364.0,372.0
1,2,357.0,194.0,226.0,128.0,76.0,120.0,71.0,91.0,136.0,144.0,392.0,352.0
2,3,385.0,199.0,152.0,179.0,107.0,121.0,118.0,75.0,136.0,154.0,468.0,313.0
3,4,343.0,242.0,129.0,109.0,113.0,171.0,144.0,105.0,136.0,176.0,415.0,310.0
4,5,340.0,244.0,142.0,135.0,183.0,166.0,96.0,85.0,121.0,177.0,453.0,297.0
5,6,400.0,265.0,142.0,142.0,234.0,139.0,72.0,101.0,106.0,211.0,421.0,285.0
6,7,377.0,285.0,173.0,142.0,173.0,239.0,77.0,102.0,105.0,215.0,395.0,320.0
7,8,375.0,144.0,215.0,172.0,131.0,161.0,71.0,110.0,84.0,164.0,426.0,325.0
8,9,434.0,212.0,119.0,217.0,197.0,152.0,64.0,118.0,55.0,174.0,437.0,322.0
9,10,407.0,187.0,187.0,195.0,203.0,134.0,65.0,133.0,46.0,179.0,278.0,314.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,259.0,164.0000,181.000000,106.000000,91.000000,96.000000,69.000000,92.000000,140.000000,146.0,364.000000,372.0
1,2,357.0,194.0000,226.000000,128.000000,76.000000,120.000000,71.000000,91.000000,136.000000,144.0,392.000000,352.0
2,3,385.0,199.0000,152.000000,179.000000,107.000000,121.000000,118.000000,75.000000,136.000000,154.0,468.000000,313.0
3,4,343.0,242.0000,129.000000,109.000000,113.000000,171.000000,80.205882,105.000000,136.000000,176.0,415.000000,310.0
4,5,340.0,244.0000,142.000000,135.000000,183.000000,166.000000,96.000000,85.000000,121.000000,177.0,453.000000,297.0
5,6,400.0,265.0000,142.000000,142.000000,234.000000,139.000000,72.000000,101.000000,106.000000,211.0,421.000000,285.0
6,7,377.0,285.0000,173.000000,142.000000,173.000000,118.323529,77.000000,102.000000,105.000000,215.0,395.000000,320.0
7,8,375.0,144.0000,215.000000,172.000000,131.000000,161.000000,71.000000,110.000000,84.000000,164.0,426.000000,325.0
8,9,434.0,212.0000,119.000000,217.000000,197.000000,152.000000,64.000000,118.000000,55.000000,174.0,437.000000,322.0
9,10,407.0,187.0000,187.000000,195.000000,203.000000,134.000000,65.000000,133.000000,46.000000,179.0,278.000000,314.0
